# Activation analysis — persona self-recognition

This notebook is the **analysis layer**. It consumes the activations captured by
`generate_text.py` (generation phase) and `evaluate_self_recognition.py` /
`evaluation_cases.py` (binary 12-case eval phase), joined to the trial-metadata
table. It does **not** modify the capture or evaluation phases.

**One knob:** set `RUN_NAME` below to the run you want to analyse (the `run_name`
from `self_recognition_config.yaml`, e.g. `personacat_v1`). That name is the R2
prefix both phases live under; `load_dataset` fetches and joins them.

**Storage note.** The brief calls the store "Zarr"; this repo shards into
`safetensors` (fp16) + a `metadata.parquet` table (R2-mirrored). The reader
(`analyze_activations_helpers`) abstracts that. Each capture is `[n_layers,
hidden]` fp16; we compute in fp32 and store small results.

**Conventions honoured everywhere:** activations are read at the *existing*
named positions (`final_prompt_token_before_answer` = the decision token,
`pre_text_token`, `text1/2_mean`, `generated_text_mean`, `persona_prompt_*`, …);
everything is per captured layer; persona **category** (suppression / near-twin /
calibration / confound) and case/condition are first-class grouping keys; nothing
is pooled across cases or conditions silently; contrast vectors are built on a
TRAIN split and evaluated on HELD-OUT trials; d′ and AUROC are the default
metrics with bootstrap CIs.

**Experiment map** (also in `analyze_activations_README.md`):
- *Enabling / control:* exp01 (persona vectors), exp03 (behavioral baseline),
  exp04 (nuisance directions), exp11 (cross-case mechanism).
- *Access / introspection:* exp02, exp05, exp06, exp07, exp08, exp09, exp10, exp14.
- *Depth:* exp12 (persona switching), exp13 (hidden-goal clustering).

**Build order:** exp01 and exp04 are prerequisites; exp14 (causal) is last. The
suite runner at the bottom respects this; individual cells call `dep()` to
materialise any prerequisite on demand.

In [ ]:
import sys
from pathlib import Path

# Put the repo root on the path so `core` / `experiments` import from anywhere.
ROOT = Path.cwd()
while not (ROOT / "core").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from experiments.self_recognition import analyze_activations_helpers as H
print("helpers loaded from", H.__file__)

In [ ]:
# ── THE ONE KNOB ─────────────────────────────────────────────────────────────
RUN_NAME = "personacat_v1"          # ← the run_name from self_recognition_config.yaml

cfg = H.AnalysisConfig(
    run_name=RUN_NAME,
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    task="persona_category",        # locates the local eval trial table (exp14 only)
    dry_run=True,                   # ← flip to False for the full run
    confident_threshold=0.60,
    train_frac=0.60,
    n_bootstrap=1000,
    seed=0,
    primary_case="case7",           # ideal self-rec source; auto-falls back to a present case
    comparison_case="case5",
    enable_steering=False,          # exp14 only: needs a GPU + the model + local trial table
)

STORE = {}   # results cache: later experiments consume earlier ones (exp02 needs exp04, …)

def run_cell(key, plot=None):
    '''Run one registered experiment into STORE and (optionally) plot it. No-op
    until the data is loaded, so the cell is safe to define before `ds` exists.'''
    if "ds" not in globals() or ds is None:
        print(f"(define-only: load the dataset cell first, then re-run {key})")
        return None
    STORE[key] = H.EXPERIMENTS[key]["run"](ds, cfg, STORE)
    if plot is not None:
        try:
            plot(STORE[key]); plt.show()
        except Exception as e:
            print("plot skipped:", e)
    return STORE[key]

def dep(store, key):
    '''Materialise a prerequisite experiment's result on demand (so a single cell
    can be run out of order).'''
    cur = store.get(key)
    if cur is None or (isinstance(cur, dict) and cur.get("error")):
        store[key] = H.EXPERIMENTS[key]["run"](ds, cfg, store)
    return store[key]

In [ ]:
# Fetch (from R2 if not cached) + read both phases, join to metadata, tag categories.
ds = H.load_dataset(cfg)
H.summarize(ds)

## Shared experiment-level helpers

Small pieces several experiments share but that are still *analysis* choices
(not generic plumbing): which present case stands in for the self-recognition
probe, the style direction both exp02 and exp09 project out, and a few
answer-semantics decoders that read the trial's `answer_order` to recover what
the model's A/B actually pointed at.

In [ ]:
def pick_self_case(ds, cfg):
    '''The cleanest self-recognition case PRESENT in this run. case7 (active,
    no description) is ideal; case1 (single-text, active, no description) is also
    decode-free; case12 mirrors case7 over calibration personas.'''
    present = set(ds.eval_meta["case"].unique()) if not ds.eval_meta.empty else set()
    for c in [cfg.primary_case, "case7", "case1", "case12", "case3"]:
        if c in present:
            return c
    return sorted(present)[0] if present else None


def style_direction(ds, cfg):
    '''A single, order-independent STYLE direction per layer, shared by exp02
    (project out) and exp09 (report). Defined as PC1 of the persona-behavior
    vectors across personas (generation space) — the dominant axis of surface
    stylistic variation, in the same residual space as the eval read positions.
    Documented choice; an alternative is a Case-5 third-party style contrast.'''
    gm = H.gen_view(ds, cfg, neutral_only=True)
    behav = {}
    for p, g in gm.groupby("persona"):
        arr, _ = H.stack_gen(ds, g, H.GEN_TEXT_MEAN)
        if len(arr):
            behav[p] = arr.mean(0)            # [L, H]
    if len(behav) < 2:
        return None
    M = np.stack(list(behav.values()))        # [P, L, H]
    L = M.shape[1]
    out = np.zeros((L, M.shape[2]), np.float32)
    for li in range(L):
        X = M[:, li, :] - M[:, li, :].mean(0, keepdims=True)
        _, _, vt = np.linalg.svd(X, full_matrices=False)
        out[li] = vt[0]
    return out


def multiclass_balacc(Xtr, ytr, Xte, yte):
    '''Balanced accuracy of an L2 multinomial logistic probe (multiclass targets
    where AUROC is awkward). NaN if a split lacks >=2 classes.'''
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import balanced_accuracy_score
    ytr = np.asarray(ytr); yte = np.asarray(yte)
    if len(set(ytr.tolist())) < 2 or len(set(yte.tolist())) < 2:
        return float("nan")
    sc = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=2000).fit(sc.transform(Xtr), ytr)
    return float(balanced_accuracy_score(yte, clf.predict(sc.transform(Xte))))


def _complement(key):
    return {"text1": "text2", "text2": "text1", "current": "other", "other": "current"}.get(key, key)


def _answer_key(row):
    '''The answer-semantic key the model's predicted letter pointed at, decoded
    from answer_order (e.g. 'A=text1'). Returns None if undecodable.'''
    ao = getattr(row, "answer_order", None); pred = getattr(row, "predicted_answer", None)
    if isinstance(ao, str) and "=" in ao and pred in ("A", "B"):
        akey = ao.split("=", 1)[1]
        return akey if pred == "A" else _complement(akey)
    return None


def _pointed_slot(sub):
    '''Per row: which physical text slot ('text1'/'text2') the answer pointed at,
    for paired self-cases whose keys are positions. None otherwise.'''
    out = []
    for r in sub.itertuples(index=False):
        k = _answer_key(r)
        out.append(k if k in ("text1", "text2") else None)
    return np.array(out, dtype=object)


def _chose_self(df):
    '''Per row: did the model pick its OWN text? Handles single-text keys
    ('current') and paired keys ('text1'/'text2' vs self_slot).'''
    out = []
    for r in df.itertuples(index=False):
        k = _answer_key(r); cs = False
        if k == "current":
            cs = True
        elif k in ("text1", "text2"):
            cs = (k == getattr(r, "self_slot", "none"))
        out.append(cs)
    return np.array(out, bool)


def _decision_self_dirs(ds, cfg, case):
    '''Per-persona decision-token self direction (position-entangled; exp04
    nuisance handling cleans it). Paired cases: self-in-text1 vs self-in-text2;
    single-text: self-shown vs other-shown. Built on confident-correct trials.'''
    df = H.eval_view(ds, cfg, cases=[case])
    if "confident_correct" in df:
        df = df[df["confident_correct"]]
    arr, sub = H.stack_eval(ds, df, H.DECISION)
    out = {}
    if len(arr) == 0:
        return out
    slot = sub["self_slot"].to_numpy() if "self_slot" in sub else np.array(["none"] * len(sub))
    if (slot == "none").all():
        if "true_author" in sub:
            isself = (sub["true_author"] == sub["evaluator_persona"]).to_numpy()
            for p in sub["evaluator_persona"].unique():
                pm = (sub["evaluator_persona"] == p).to_numpy()
                a, b = arr[pm & isself], arr[pm & ~isself]
                if len(a) >= 2 and len(b) >= 2:
                    out[p] = H.mean_diff(a, b)
    else:
        for p in sub["evaluator_persona"].unique():
            pm = (sub["evaluator_persona"] == p).to_numpy()
            a, b = arr[pm & (slot == "text1")], arr[pm & (slot == "text2")]
            if len(a) >= 2 and len(b) >= 2:
                out[p] = H.mean_diff(a, b)
    return out


def _messages_from_row(r):
    '''Reconstruct the chat messages for one trial from the local trial table's
    exact prompt_text / system_prompt_text (exp14).'''
    sysp = getattr(r, "system_prompt_text", None); usr = getattr(r, "prompt_text", "")
    msgs = []
    if isinstance(sysp, str) and sysp.strip():
        msgs.append({"role": "system", "content": sysp})
    msgs.append({"role": "user", "content": usr})
    return msgs

## exp01 — Persona vectors (building block)

Two vectors per persona per layer:
- **persona_behavior_vector** = mean-pooled `generated_text_mean` on NEUTRAL
  generation tasks. For clustering / alignment (exp05, exp10, exp13).
- **persona_prompt_state_vector** = mean of `persona_prompt_{mean,final}`.
  **Prompt-token-exposed** — use only for the "stays in persona" drift analysis
  (exp12), never as a self-recognition signal.

In [ ]:
@H.register("exp01", "Persona vectors: behavior (neutral gen) + prompt-state")
def exp01(ds, cfg, store):
    gm = H.gen_view(ds, cfg, neutral_only=True)
    H.report_cells(gm, ["persona", "persona_coarse"], "exp01 neutral-gen rows per persona")
    layers = ds.gen.layers
    behavior, prompt_state, n_rows = {}, {}, {}
    for p in sorted(gm["persona"].unique()):
        g = gm[gm["persona"] == p]
        b, _ = H.stack_gen(ds, g, H.GEN_TEXT_MEAN)
        pm, _ = H.stack_gen(ds, g, H.PERSONA_PROMPT_MEAN)
        pf, _ = H.stack_gen(ds, g, H.PERSONA_PROMPT_FINAL)
        if len(b) == 0:
            continue
        behavior[p] = b.mean(0)
        comps = [a.mean(0) for a in (pm, pf) if len(a)]
        if comps:
            prompt_state[p] = np.mean(comps, axis=0)
        n_rows[p] = int(len(b))
    order = [p for p in behavior]
    cats = {p: ds.persona_category.get(p, "") for p in order}
    d = H.art_dir(cfg, "exp01")
    if order:
        H.save_npz(d / "persona_vectors.npz",
                   behavior=np.stack([behavior[p] for p in order]),
                   personas=np.array(order), layers=np.array(layers),
                   prompt_state=np.stack([prompt_state[p] for p in order if p in prompt_state])
                                if any(p in prompt_state for p in order) else np.zeros((0,)),
                   prompt_personas=np.array([p for p in order if p in prompt_state]))
    H.save_json({"personas": order, "n_rows": n_rows, "categories": cats, "layers": layers,
                 "note": "behavior=mean generated_text_mean on neutral tasks (clustering/alignment); "
                         "prompt_state=mean persona_prompt_{mean,final} (prompt-token exposed; exp12 only)"},
                d / "exp01.json")
    return {"behavior": behavior, "prompt_state": prompt_state, "personas": order,
            "layers": layers, "categories": cats, "n_rows": n_rows}


def plot_exp01(res):
    behav, layers = res["behavior"], res["layers"]
    if len(behav) < 2:
        print("exp01: <2 personas"); return
    personas = list(behav)
    shared = []
    for li in range(len(layers)):
        C = H.cosine_matrix(np.stack([behav[p][li] for p in personas]))
        shared.append(C[~np.eye(len(personas), dtype=bool)].mean())
    H.plot_layer_curves(layers, {"mean pairwise cosine": shared},
                        ylabel="cosine", title="exp01: persona-behavior vector overlap by layer", hline=0.0)


run_cell("exp01", plot_exp01)

## exp04 — Nuisance directions (building block; prerequisite)

Decision-token directions for the answer nuisances present in this A/B design:
**A/B** (emit-A vs emit-B) and **text-position** (answer points to physical
text1 vs text2). Yes/No and answer-format directions don't exist in the binary
A/B cases — reported as absent rather than faked.

`H.project_out` returns BOTH the raw and nuisance-removed version of any vector —
nothing is auto-subtracted. Before any out-projection, exp02/exp09 report the
cosine of each self-rec direction against these nuisances.

In [ ]:
@H.register("exp04", "Nuisance directions (A/B, text-position) at the decision token")
def exp04(ds, cfg, store):
    df = H.eval_view(ds, cfg)
    layers = ds.eval.layers
    arr, sub = H.stack_eval(ds, df, H.DECISION)
    H.report_cells(sub, ["case", "predicted_answer"], "exp04 decision-token rows")
    dirs = {}
    if len(arr):
        a = arr[(sub["predicted_answer"] == "A").to_numpy()]
        b = arr[(sub["predicted_answer"] == "B").to_numpy()]
        if len(a) >= 2 and len(b) >= 2:
            dirs["AB"] = H.mean_diff(a, b)
        slot = _pointed_slot(sub)
        m1, m2 = (slot == "text1"), (slot == "text2")
        if m1.sum() >= 2 and m2.sum() >= 2:
            dirs["text_pos"] = H.mean_diff(arr[m1], arr[m2])
    note = ("Yes/No and answer-format directions absent in A/B binary cases; "
            "built A/B (answer-letter) + text-position nuisance directions.")
    cos = {}
    e2 = store.get("exp02")
    srd = e2.get("pooled_read_dir") if isinstance(e2, dict) else None
    if srd is not None:
        for nm, dv in dirs.items():
            cos[nm] = [float(H.cosine(dv[li], srd[li])) for li in range(len(layers))]
    d = H.art_dir(cfg, "exp04")
    H.save_npz(d / "nuisance_dirs.npz", layers=np.array(layers), **dirs)
    H.save_json({"directions": list(dirs), "note": note, "cosine_vs_self_rec": cos}, d / "exp04.json")
    print("nuisance directions:", list(dirs))
    if cos:
        print("cosine vs self-rec (per layer):", {k: [round(x, 3) for x in v] for k, v in cos.items()})
    else:
        print("(run exp02, then re-run exp04 to report cosine vs the self-rec direction)")
    return {"dirs": dirs, "layers": layers, "cosine_vs_self_rec": cos, "note": note}


def plot_exp04(res):
    if not res.get("cosine_vs_self_rec"):
        print("exp04: cosine-vs-self-rec available after exp02 — directions:", list(res["dirs"])); return
    H.plot_layer_curves(res["layers"], {k: v for k, v in res["cosine_vs_self_rec"].items()},
                        ylabel="cosine", title="exp04: nuisance vs self-rec direction", hline=0.0)


run_cell("exp04", plot_exp04)

## exp02 — Self-recognition vector, with / without style removed (headline)

Per persona, the self-recognition direction is built two ways and saved for
exp05/exp14:
- **read-side** (`read_dirs`): mean(reading self-authored text span) −
  mean(reading other-authored span), confident-correct trials only, from the
  cleanest present case (case7 if available; auto-fallback). Decode-free in
  case7 by construction. This is the projectable direction exp06/exp07 use.
- **decision-token** (`decision_dirs`): self-in-text1 vs self-in-text2 (paired)
  or self-shown vs other-shown (single-text). Position-entangled — that's why
  exp04 is a prerequisite.

**Headline:** held-out AUROC of the read direction separating self/other text,
*raw* vs with the exp09-consistent **style direction projected out** (via the
exp04 helper; never auto-removed). Does the direction survive style removal?

In [ ]:
@H.register("exp02", "Self-recognition vector (read-side + decision), style-removed variant")
def exp02(ds, cfg, store):
    case = pick_self_case(ds, cfg)
    print("self-recognition source case:", case)
    layers = ds.eval.layers
    df = H.eval_view(ds, cfg, cases=[case])
    if "confident_correct" in df:
        df = df[df["confident_correct"]]
    H.report_cells(df, ["case", "evaluator_coarse"], f"exp02 {case} confident-correct trials")
    V, is_self, persona, base = H.authorship_samples(ds, df)
    style = style_direction(ds, cfg)

    read_dirs, pooled = {}, None
    auroc = {"raw": [], "style_removed": []}
    bands = {"raw": ([], []), "style_removed": ([], [])}
    if len(V):
        tr, te = H.split_by_groups(base, cfg.train_frac, cfg.seed)
        for p in np.unique(persona):
            pm = persona == p
            sp, op = V[pm & is_self & tr], V[pm & ~is_self & tr]
            if len(sp) >= 2 and len(op) >= 2:
                read_dirs[p] = H.mean_diff(sp, op)
        sup = np.array([H.coarse_category(ds.persona_category.get(p, "")) == "suppression" for p in persona])
        pool = sup if sup.sum() >= 4 else np.ones(len(persona), bool)
        sp, op = V[pool & is_self & tr], V[pool & ~is_self & tr]
        if len(sp) >= 2 and len(op) >= 2:
            pooled = H.mean_diff(sp, op)
        for li in range(len(layers)):
            if pooled is None or te.sum() == 0:
                for k in auroc:
                    auroc[k].append(float("nan")); bands[k][0].append(float("nan")); bands[k][1].append(float("nan"))
                continue
            Xte, yte = V[te][:, li, :], is_self[te]
            c1 = H.bootstrap_ci(lambda y, s: H.auroc(y, s), (yte, H.proj_scalar(Xte, pooled[li])),
                                n=cfg.n_bootstrap, seed=cfg.seed)
            auroc["raw"].append(c1["point"]); bands["raw"][0].append(c1["lo"]); bands["raw"][1].append(c1["hi"])
            if style is not None:
                d_sr = H.project_out_vec(pooled[li], style[li])
                s_sr = H.proj_scalar(H.project_out(Xte, style[li]), d_sr)
                c2 = H.bootstrap_ci(lambda y, s: H.auroc(y, s), (yte, s_sr), n=cfg.n_bootstrap, seed=cfg.seed)
                auroc["style_removed"].append(c2["point"]); bands["style_removed"][0].append(c2["lo"]); bands["style_removed"][1].append(c2["hi"])
            else:
                auroc["style_removed"].append(float("nan")); bands["style_removed"][0].append(float("nan")); bands["style_removed"][1].append(float("nan"))

    # Before any out-projection: report cosine of the self-rec direction against
    # the exp04 nuisance directions (exp04 precedes exp02 in the suite order).
    e4 = store.get("exp04"); cos_nuis = {}
    if isinstance(e4, dict) and e4.get("dirs") and pooled is not None:
        cos_nuis = {nm: [float(H.cosine(pooled[li], dv[li])) for li in range(len(layers))]
                    for nm, dv in e4["dirs"].items()}
        print("cosine(self-rec, nuisance) per layer:",
              {k: [round(x, 3) for x in v] for k, v in cos_nuis.items()})

    decision_dirs = _decision_self_dirs(ds, cfg, case)
    d = H.art_dir(cfg, "exp02")
    if read_dirs:
        H.save_npz(d / "self_rec_read_dirs.npz", layers=np.array(layers),
                   personas=np.array(list(read_dirs)),
                   read_dirs=np.stack([read_dirs[p] for p in read_dirs]),
                   pooled=pooled if pooled is not None else np.zeros((0,)))
    H.save_json({"case": case, "personas": list(read_dirs), "layers": layers,
                 "auroc_raw": auroc["raw"], "auroc_style_removed": auroc["style_removed"],
                 "cosine_vs_nuisance": cos_nuis,
                 "survives_style": bool(style is not None and len(auroc["style_removed"])
                                        and np.nanmax(auroc["style_removed"]) > 0.6)},
                d / "exp02.json")
    return {"case": case, "read_dirs": read_dirs, "pooled_read_dir": pooled,
            "decision_dirs": decision_dirs, "layers": layers, "auroc": auroc,
            "bands": bands, "style_dir": style, "cosine_vs_nuisance": cos_nuis,
            "samples": {"V": V, "is_self": is_self, "persona": persona, "base": base}}


def plot_exp02(res):
    if not any(np.isfinite(res["auroc"]["raw"])):
        print("exp02: no held-out AUROC (too few trials in dry-run?)"); return
    H.plot_layer_curves(res["layers"],
                        {"raw": res["auroc"]["raw"], "style-removed": res["auroc"]["style_removed"]},
                        ylabel="held-out AUROC (self vs other)",
                        title=f"exp02: self-rec direction survives style removal? ({res['case']})",
                        hline=0.5, bands={"raw": res["bands"]["raw"], "style-removed": res["bands"]["style_removed"]})


run_cell("exp02", plot_exp02)
# exp04's cosine-vs-self-rec is available now that exp02 exists:
if "ds" in globals() and ds is not None:
    run_cell("exp04", plot_exp04)

## exp03 — Behavioral recognition baseline (enabling / control)

*(This number, exp03, is referenced in the brief's "enabling/control" group but
its body was not specified; implemented here as the behavioral table the
activation experiments are read against.)*

Accuracy (with bootstrap CI) and mean decision confidence by **case ×
system-prompt-present × persona coarse-category**, straight from the trial table.
Tells you which cases/personas actually recognize — the rows where an activation
signal should exist.

In [ ]:
@H.register("exp03", "Behavioral recognition baseline (accuracy/confidence by case x condition x category)")
def exp03(ds, cfg, store):
    df = H.eval_view(ds, cfg)
    H.report_cells(df, ["case", "system_prompt_present", "evaluator_coarse"], "exp03 cells")
    rows = []
    for (case, sp, coarse), g in df.groupby(["case", "system_prompt_present", "evaluator_coarse"]):
        corr = g["correct"].fillna(False).astype(bool).to_numpy()
        ci = H.bootstrap_ci(lambda c: float(c.mean()), (corr,), n=cfg.n_bootstrap, seed=cfg.seed)
        rows.append({"case": case, "system_prompt_present": bool(sp), "coarse": coarse, "n": int(len(g)),
                     "accuracy": ci["point"], "acc_lo": ci["lo"], "acc_hi": ci["hi"],
                     "mean_decision_conf": float(g["decision_conf"].mean()),
                     "mean_prob_correct": float(g["answer_confidence"].mean()) if "answer_confidence" in g else float("nan")})
    out = pd.DataFrame(rows).sort_values(["case", "coarse"]).reset_index(drop=True)
    H.save_df(out, H.art_dir(cfg, "exp03") / "behavioral_baseline.parquet")
    print(out.to_string(index=False))
    return {"table": out}


def plot_exp03(res):
    out = res["table"]
    if out.empty:
        return
    sub = out[out["system_prompt_present"]] if out["system_prompt_present"].any() else out
    labels = [f"{r.case}/{r.coarse}" for r in sub.itertuples()]
    errs = [[max(0, r.accuracy - r.acc_lo) for r in sub.itertuples()],
            [max(0, r.acc_hi - r.accuracy) for r in sub.itertuples()]]
    H.plot_bars(labels, sub["accuracy"].tolist(), errs, ylabel="accuracy",
                title="exp03: recognition accuracy by case x coarse-category", hline=0.5)


run_cell("exp03", plot_exp03)

## exp05 — Are self-rec vectors persona-specific or shared? (access)

Pairwise cosine of exp02 read directions across personas (shared vs distinct,
per layer), and per persona the cosine between its self-rec direction and its own
exp01 persona-behavior vector ("is self-recognition aligned with detecting my own
persona signature?"). Suppression personas are the rows that matter.

In [ ]:
@H.register("exp05", "Self-rec vectors: shared vs distinct + alignment to persona behavior")
def exp05(ds, cfg, store):
    e1 = dep(store, "exp01"); e2 = dep(store, "exp02")
    read, behav, layers = e2["read_dirs"], e1["behavior"], e2["layers"]
    personas = [p for p in read if p in behav]
    if len(personas) < 2:
        print("exp05: <2 personas with both vectors"); return {"personas": personas}
    shared, cosmats = [], {}
    for li in range(len(layers)):
        M = np.stack([read[p][li] for p in personas])
        C = H.cosine_matrix(M); cosmats[li] = C
        shared.append(float(C[~np.eye(len(personas), dtype=bool)].mean()))
    align = {p: [float(H.cosine(read[p][li], behav[p][li])) for li in range(len(layers))] for p in personas}
    sup = [p for p in personas if H.coarse_category(ds.persona_category.get(p, "")) == "suppression"]
    print("personas:", personas, "\nsuppression rows:", sup)
    H.save_json({"personas": personas, "layers": layers, "shared_cosine": shared,
                 "alignment": align, "suppression": sup}, H.art_dir(cfg, "exp05") / "exp05.json")
    return {"personas": personas, "layers": layers, "shared_cosine": shared,
            "alignment": align, "cosmats": cosmats, "suppression": sup}


def plot_exp05(res):
    if "shared_cosine" not in res:
        return
    layers = res["layers"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    sup = set(res["suppression"])
    H.plot_layer_curves(layers, {"all (mean pairwise)": res["shared_cosine"]},
                        ylabel="cosine", title="exp05: shared-ness of self-rec dirs", hline=0.0, ax=ax[0])
    series = {p: res["alignment"][p] for p in res["alignment"] if p in sup} or res["alignment"]
    H.plot_layer_curves(layers, series, ylabel="cosine(self-rec, behavior)",
                        title="exp05: alignment to own persona signature", hline=0.0, ax=ax[1])
    fig.tight_layout()


run_cell("exp05", plot_exp05)

## exp06 — Self-prior vs text-sensitive evidence (access)

Project the residual onto the persona's self-rec direction (exp02) at the
**pre-text** position (prior / bias) and at the **decision** position (after
reading). Per persona: prior level vs the evidence-driven increase
(decision − pre-text), with bootstrap CIs. Headline: do personas with high
false-self selection sit high *before* reading (bias) or only rise *after*
(uptake)?

In [ ]:
@H.register("exp06", "Self-prior (pre-text) vs evidence-driven (decision) projection")
def exp06(ds, cfg, store):
    e2 = dep(store, "exp02"); read, layers, case = e2["read_dirs"], e2["layers"], e2["case"]
    df = H.eval_view(ds, cfg, cases=[case])
    H.report_cells(df, ["case", "evaluator_coarse"], "exp06 trials")

    def proj_df(cap, tag):
        arr, sub = H.stack_eval(ds, df, cap)
        cols = {f"{tag}_L{l}": np.full(len(sub), np.nan) for l in layers}
        for i, r in enumerate(sub.itertuples(index=False)):
            ev = r.evaluator_persona
            if ev in read:
                for li, l in enumerate(layers):
                    cols[f"{tag}_L{l}"][i] = float(arr[i, li] @ H.unit(read[ev][li]))
        out = pd.DataFrame(cols)
        out["id"] = sub["id"].to_numpy(); out["evaluator_persona"] = sub["evaluator_persona"].to_numpy()
        return out

    pre, dec = proj_df(H.PRE_TEXT, "pre"), proj_df(H.DECISION, "dec")
    m = pre.merge(dec, on=["id", "evaluator_persona"], how="inner")
    # false-self rate per persona from the full eval table
    ev = ds.eval_meta[ds.eval_meta["case"] == case].copy()
    ev["chose_self"] = _chose_self(ev)
    fsr = ev.assign(falsely=(ev["chose_self"] & ~ev["correct"].fillna(False))).groupby(
        "evaluator_persona")["falsely"].mean().to_dict()
    rows = []
    for ev_name, g in m.groupby("evaluator_persona"):
        coarse = H.coarse_category(ds.persona_category.get(ev_name, ""))
        for li, l in enumerate(layers):
            pr, de = g[f"pre_L{l}"].to_numpy(), g[f"dec_L{l}"].to_numpy()
            ok = np.isfinite(pr) & np.isfinite(de)
            if ok.sum() < 2:
                continue
            rows.append({"evaluator": ev_name, "coarse": coarse, "layer": l, "n": int(ok.sum()),
                         "prior": float(pr[ok].mean()), "decision": float(de[ok].mean()),
                         "delta": float((de[ok] - pr[ok]).mean()), "false_self_rate": float(fsr.get(ev_name, np.nan))})
    out = pd.DataFrame(rows)
    H.save_df(out, H.art_dir(cfg, "exp06") / "prior_vs_evidence.parquet")
    return {"layers": layers, "case": case, "table": out}


def plot_exp06(res):
    out = res["table"]
    if out.empty:
        print("exp06: no rows"); return
    g = out.groupby(["coarse", "layer"]).agg(prior=("prior", "mean"), delta=("delta", "mean")).reset_index()
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for coarse, gg in g.groupby("coarse"):
        ax[0].plot(gg["layer"], gg["prior"], marker="o", label=coarse)
        ax[1].plot(gg["layer"], gg["delta"], marker="o", label=coarse)
    for a, t in zip(ax, ["prior (pre-text projection)", "evidence uptake (decision - pre)"]):
        a.axhline(0, ls="--", c="grey", lw=1); a.set_xlabel("layer"); a.set_title("exp06: " + t); a.legend(fontsize=8)
    fig.tight_layout()


run_cell("exp06", plot_exp06)

## exp07 — Online vs decision-time recognition (layer × token map) (access)

Track the self-rec projection across captured positions (pre-text → text1 →
text2 → decision), swept over layers, per persona category. Text-span
projections are **signed by authorship** (self-authored = +): because text
position is counterbalanced, a signal that rises while reading self-authored text
*follows the self text across the swap* (introspection), whereas a first-read
artifact would wash out. Rising-during-reading vs flat-until-answer is the
introspection-vs-confabulation readout.

In [ ]:
@H.register("exp07", "Online vs decision-time recognition (layer x token map)")
def exp07(ds, cfg, store):
    e2 = dep(store, "exp02"); read, layers, case = e2["read_dirs"], e2["layers"], e2["case"]
    df = H.eval_view(ds, cfg, cases=[case])
    positions = [("pre_text", H.PRE_TEXT), ("text1", H.TEXT1_MEAN),
                 ("text2", H.TEXT2_MEAN), ("decision", H.DECISION)]
    cats = ["suppression", "near_twin", "calibration", "confound"]
    maps = {}
    for coarse in cats:
        evs = [p for p in read if H.coarse_category(ds.persona_category.get(p, "")) == coarse]
        sub_df = df[df["evaluator_persona"].isin(evs)]
        if len(sub_df) == 0:
            continue
        M = np.full((len(layers), len(positions)), np.nan)
        for pi, (pname, cap) in enumerate(positions):
            arr, s = H.stack_eval(ds, sub_df, cap)
            if len(arr) == 0:
                continue
            acc = [[] for _ in layers]
            for i, r in enumerate(s.itertuples(index=False)):
                ev = r.evaluator_persona
                if ev not in read:
                    continue
                sign = 1.0
                if pname in ("text1", "text2"):
                    src = getattr(r, {"text1": "text1_source_persona", "text2": "text2_source_persona"}[pname], None)
                    sign = 1.0 if src == ev else -1.0
                for li in range(len(layers)):
                    acc[li].append(sign * float(arr[i, li] @ H.unit(read[ev][li])))
            for li in range(len(layers)):
                if acc[li]:
                    M[li, pi] = float(np.mean(acc[li]))
        maps[coarse] = M
    H.save_npz(H.art_dir(cfg, "exp07") / "layer_token_maps.npz", layers=np.array(layers),
               positions=np.array([p for p, _ in positions]),
               **{c: maps[c] for c in maps})
    return {"layers": layers, "positions": [p for p, _ in positions], "maps": maps, "case": case}


def plot_exp07(res):
    maps = res["maps"]
    if not maps:
        print("exp07: no category maps"); return
    n = len(maps)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), squeeze=False)
    for ax, (coarse, M) in zip(axes[0], maps.items()):
        H.plot_heatmap(M, xticks=res["positions"], yticks=res["layers"],
                       title=f"exp07: {coarse}", xlabel="position", ylabel="layer",
                       cmap="coolwarm", ax=ax, annotate=True)
    fig.tight_layout()


run_cell("exp07", plot_exp07)

## exp08 — Where authorship vs style is encoded, by layer (access)

Linear probes on read-side text-span means decoding, separately: **true author**
(self/other, AUROC), **persona identity** (who authored, balanced accuracy),
**task family** (balanced accuracy), and **text position** (text1/text2, AUROC).
Train/test split by `base_trial_id`. Headline: layers where author is decodable
but persona-identity / style is *not* (or less) = candidate access locus.

In [ ]:
@H.register("exp08", "Probes: author vs persona-identity vs task vs position by layer")
def exp08(ds, cfg, store):
    df = H.eval_view(ds, cfg)
    V, tab = H.span_samples(ds, df)
    if len(V) == 0:
        print("exp08: no spans"); return {"layers": ds.eval.layers, "curves": {}}
    H.report_cells(tab, ["case", "is_self"], "exp08 read-side spans")
    layers = ds.eval.layers
    tr, te = H.split_by_groups(tab["base_trial_id"].to_numpy(), cfg.train_frac, cfg.seed)
    targets = {
        "author_self (AUROC)": ("bin", tab["is_self"].astype(int).to_numpy()),
        "persona_identity (balacc)": ("multi", tab["author"].astype(str).to_numpy()),
        "task_family (balacc)": ("multi", tab["task_category"].astype(str).to_numpy()),
        "text_position (AUROC)": ("bin", (tab["slot"] == "text1").astype(int).to_numpy()),
    }
    curves = {}
    for name, (kind, y) in targets.items():
        ys = []
        for li in range(len(layers)):
            X = V[:, li, :]
            ys.append(H.logistic_auroc(X[tr], y[tr], X[te], y[te]) if kind == "bin"
                      else multiclass_balacc(X[tr], y[tr], X[te], y[te]))
        curves[name] = ys
    H.save_json({"layers": layers, "curves": curves}, H.art_dir(cfg, "exp08") / "exp08.json")
    return {"layers": layers, "curves": curves}


def plot_exp08(res):
    if not res.get("curves"):
        return
    H.plot_layer_curves(res["layers"], res["curves"], ylabel="held-out score",
                        title="exp08: decodability by layer (author vs style/identity)", hline=0.5)


run_cell("exp08", plot_exp08)

## exp09 — Confabulation vs genuine recognition (access)

Among **confident self-claims** (model picked its own text, confidently), the
decision-token contrast correct vs incorrect → a "genuine recognition"
direction; AUROC per layer on held-out. Confident-wrong self-claims (confabulation)
are sparse — n-per-cell is printed and CIs widen accordingly. Reports the cosine
of the genuine direction against the shared style direction (same one exp02
projects out) and the exp02 read direction.

In [ ]:
@H.register("exp09", "Confabulation vs genuine recognition (decision-token)")
def exp09(ds, cfg, store):
    case = pick_self_case(ds, cfg); layers = ds.eval.layers
    df = H.eval_view(ds, cfg, cases=[case]).copy()
    df["chose_self"] = _chose_self(df)
    claims = df[df["chose_self"] & df["is_confident"]]
    H.report_cells(claims, ["case", "correct"], "exp09 confident self-claims (genuine vs confab)")
    arr, sub = H.stack_eval(ds, claims, H.DECISION)
    genuine_dir, auroc, lo, hi = None, [], [], []
    if len(arr):
        ok = (sub["correct"] == True).to_numpy(); bad = ~ok
        print(f"n correct-confident={int(ok.sum())}  n incorrect-confident(confab)={int(bad.sum())}")
        if ok.sum() >= 2 and bad.sum() >= 2:
            tr, te = H.split_by_groups(sub["base_trial_id"].to_numpy(), cfg.train_frac, cfg.seed)
            if (ok & tr).sum() >= 2 and (bad & tr).sum() >= 2:
                genuine_dir = H.mean_diff(arr[ok & tr], arr[bad & tr])
            else:
                genuine_dir = H.mean_diff(arr[ok], arr[bad]); te = np.ones(len(arr), bool)
            for li in range(len(layers)):
                c = H.bootstrap_ci(lambda y, s: H.auroc(y, s),
                                   (ok[te], H.proj_scalar(arr[te][:, li, :], genuine_dir[li])),
                                   n=cfg.n_bootstrap, seed=cfg.seed)
                auroc.append(c["point"]); lo.append(c["lo"]); hi.append(c["hi"])
    style = style_direction(ds, cfg)
    e2 = store.get("exp02"); rd = e2.get("pooled_read_dir") if isinstance(e2, dict) else None
    cos_style = ([float(H.cosine(genuine_dir[li], style[li])) for li in range(len(layers))]
                 if genuine_dir is not None and style is not None else [])
    cos_read = ([float(H.cosine(genuine_dir[li], rd[li])) for li in range(len(layers))]
                if genuine_dir is not None and rd is not None else [])
    d = H.art_dir(cfg, "exp09")
    if genuine_dir is not None:
        H.save_npz(d / "genuine_dir.npz", layers=np.array(layers), genuine_dir=genuine_dir,
                   style_dir=style if style is not None else np.zeros((0,)))
    H.save_json({"case": case, "layers": layers, "auroc": auroc, "cos_style": cos_style,
                 "cos_read": cos_read}, d / "exp09.json")
    return {"case": case, "layers": layers, "genuine_dir": genuine_dir, "style_dir": style,
            "auroc": auroc, "bands": (lo, hi), "cos_style": cos_style, "cos_read": cos_read}


def plot_exp09(res):
    if not res["auroc"]:
        print("exp09: too few confident self-claims to build a direction"); return
    H.plot_layer_curves(res["layers"], {"genuine vs confab (AUROC)": res["auroc"]},
                        ylabel="held-out AUROC", title=f"exp09: genuine recognition direction ({res['case']})",
                        hline=0.5, bands={"genuine vs confab (AUROC)": res["bands"]})


run_cell("exp09", plot_exp09)

## exp10 — Cross-task vector generalization (access)

Build the self-recognition probe on ONE task family and test it on HELD-OUT
families (within-family vs cross-family AUROC of self/other). The **transfer
gap** (within − cross) is the headline: within-only ⇒ task/style-specific;
transfer ⇒ a robust persona/self mechanism. (Few tasks per family in this run —
n-per-cell is printed; treat small cells cautiously.)

In [ ]:
@H.register("exp10", "Cross-task vector generalization (within vs cross family)")
def exp10(ds, cfg, store):
    case = pick_self_case(ds, cfg); layers = ds.eval.layers
    df = H.eval_view(ds, cfg, cases=[case])
    if "confident_correct" in df:
        df = df[df["confident_correct"]]
    V, tab = H.span_samples(ds, df)
    if len(V) == 0:
        print("exp10: no spans"); return {"layers": layers, "families": [], "results": {}}
    H.report_cells(tab, ["task_category", "is_self"], "exp10 spans by task family")
    counts = tab["task_category"].value_counts()
    fams = [f for f in counts.index if f and counts[f] >= 10]
    y = tab["is_self"].astype(int).to_numpy()
    results = {}
    for f in fams:
        infam = (tab["task_category"] == f).to_numpy(); idx = np.where(infam)[0]
        tri, tei = H.split_by_groups(tab.loc[infam, "base_trial_id"].to_numpy(), cfg.train_frac, cfg.seed)
        within, cross = [], []
        for li in range(len(layers)):
            X = V[:, li, :]
            within.append(H.logistic_auroc(X[idx[tri]], y[idx[tri]], X[idx[tei]], y[idx[tei]]))
            cross.append(H.logistic_auroc(X[infam], y[infam], X[~infam], y[~infam]))
        gap = [(w - c) if (w == w and c == c) else float("nan") for w, c in zip(within, cross)]
        results[f] = {"within": within, "cross": cross, "gap": gap}
    H.save_json({"case": case, "layers": layers, "families": fams, "results": results},
                H.art_dir(cfg, "exp10") / "exp10.json")
    if results:
        mg = np.nanmean([np.nanmean(results[f]["gap"]) for f in results])
        print(f"mean transfer gap (within - cross), over families: {mg:.3f}")
    return {"layers": layers, "families": fams, "results": results}


def plot_exp10(res):
    if not res["results"]:
        print("exp10: no families with enough spans"); return
    L = res["layers"]
    within = np.nanmean([res["results"][f]["within"] for f in res["results"]], axis=0)
    cross = np.nanmean([res["results"][f]["cross"] for f in res["results"]], axis=0)
    H.plot_layer_curves(L, {"within-family": within.tolist(), "cross-family": cross.tolist()},
                        ylabel="AUROC (self vs other)", title="exp10: cross-task generalization", hline=0.5)


run_cell("exp10", plot_exp10)

## exp11 — Cross-case mechanism comparison (control)

After exp04 nuisance handling, compare decision-token geometry across the cases
present (self cases 1/3/7/12 vs the third-party case 5). Cosine /
representational-similarity between nuisance-cleaned case centroids per layer.
Does "which is mine" share a mechanism with "which non-me persona wrote which"?
Reports self-vs-third-party separation.

In [ ]:
@H.register("exp11", "Cross-case mechanism comparison (self vs third-party centroids)")
def exp11(ds, cfg, store):
    e4 = dep(store, "exp04"); layers = ds.eval.layers; nuis = e4["dirs"]
    df = H.eval_view(ds, cfg)
    present = sorted(df["case"].unique())
    cents = {}
    for c in present:
        arr, _ = H.stack_eval(ds, df[df["case"] == c], H.DECISION)
        if len(arr) == 0:
            continue
        clean = arr.copy()
        for li in range(len(layers)):
            for dv in nuis.values():
                clean[:, li, :] = H.project_out(clean[:, li, :], dv[li])
        cents[c] = clean.mean(0)
    cases = list(cents)
    print("cases present:", cases)
    li_rep = len(layers) // 2
    M = np.array([[float(H.cosine(cents[a][li_rep], cents[b][li_rep])) for b in cases] for a in cases])
    self_cases = [c for c in cases if c in ("case1", "case3", "case7", "case12")]
    third = [c for c in cases if c == "case5"]
    sep = None
    if self_cases and third:
        within = [M[cases.index(a), cases.index(b)] for a in self_cases for b in self_cases if a != b]
        across = [M[cases.index(a), cases.index(b)] for a in self_cases for b in third]
        sep = {"within_self": float(np.mean(within)) if within else float("nan"),
               "self_vs_third": float(np.mean(across)) if across else float("nan")}
        print("self-vs-third-party centroid cosine:", sep)
    H.save_npz(H.art_dir(cfg, "exp11") / "case_centroids.npz", layers=np.array(layers),
               cases=np.array(cases), cosine_matrix=M, **{f"cent_{c}": cents[c] for c in cases})
    H.save_json({"cases": cases, "rep_layer": layers[li_rep], "separation": sep},
                H.art_dir(cfg, "exp11") / "exp11.json")
    return {"cases": cases, "layers": layers, "cosine_matrix": M, "rep_layer": layers[li_rep],
            "cents": cents, "separation": sep}


def plot_exp11(res):
    if len(res["cases"]) < 2:
        print("exp11: <2 cases present"); return
    H.plot_heatmap(res["cosine_matrix"], xticks=res["cases"], yticks=res["cases"],
                   title=f"exp11: decision-token case centroids (cosine, layer {res['rep_layer']})",
                   annotate=True, cmap="coolwarm", vmin=-1, vmax=1)


run_cell("exp11", plot_exp11)

## exp12 — Persona switching during the task (depth, not access)

While reading the **Other-authored** text span, project the residual onto the
evaluator's own persona-behavior vector vs the Other's (exp01). Does the active
evaluator drift toward the Other while reading it? **Depth/stability readout, not
an access metric.** (True per-token drift wants the optional per-token capture
subset, which this run lacks; we use the available span-mean positions.)

In [ ]:
@H.register("exp12", "Persona switching during the task (drift toward Other)")
def exp12(ds, cfg, store):
    e1 = dep(store, "exp01"); behav = e1["behavior"]; layers = ds.gen.layers
    present = set(ds.eval_meta["case"].unique()) if not ds.eval_meta.empty else set()
    cases = [c for c in ["case5", "case8", "case9", "case3", "case12", "case7"] if c in present]
    df = H.eval_view(ds, cfg, cases=cases) if cases else H.eval_view(ds, cfg)
    nL = min(len(layers), min((behav[p].shape[0] for p in behav), default=0))
    rows = []
    for r in df.itertuples(index=False):
        ev = r.evaluator_persona
        if ev not in behav:
            continue
        for slot, cap, srccol in (("text1", H.TEXT1_MEAN, "text1_source_persona"),
                                  ("text2", H.TEXT2_MEAN, "text2_source_persona")):
            src = getattr(r, srccol, None)
            if src is None or src == ev or src not in behav:
                continue
            v = ds.eval.vector(r.id, cap)
            if v is None:
                continue
            for li in range(nL):
                own = float(v[li] @ H.unit(behav[ev][li])); oth = float(v[li] @ H.unit(behav[src][li]))
                rows.append({"layer": layers[li], "evaluator": ev, "other": src,
                             "own_proj": own, "other_proj": oth, "drift": oth - own})
    out = pd.DataFrame(rows)
    H.report_cells(out, ["layer"], "exp12 other-text-span projections", max_rows=12)
    H.save_df(out, H.art_dir(cfg, "exp12") / "switching.parquet") if not out.empty else None
    return {"layers": layers[:nL], "table": out}


def plot_exp12(res):
    out = res["table"]
    if out.empty:
        print("exp12: no other-authored spans (need described/2-text cases)"); return
    g = out.groupby("layer").agg(drift=("drift", "mean")).reset_index()
    H.plot_layer_curves(g["layer"].tolist(), {"drift toward Other (other - own)": g["drift"].tolist()},
                        ylabel="projection difference", title="exp12: drift while reading the Other", hline=0.0)


run_cell("exp12", plot_exp12)

## exp13 — Deceptive / hidden-goal clustering (depth)

Cluster the exp01 persona-behavior vectors. Do confound / hidden-goal personas
(sandbagger, sycophant) occupy a shared region independent of surface text? The
non-deceptive **historians** are the control cluster (rules out "any grouping").
Deliberately independent of recognition scores — no accuracy here.

In [ ]:
@H.register("exp13", "Hidden-goal / deceptive clustering of persona behavior vectors")
def exp13(ds, cfg, store):
    e1 = dep(store, "exp01"); behav = e1["behavior"]; layers = e1["layers"]
    personas = list(behav)
    if len(personas) < 3:
        print("exp13: too few personas"); return {"personas": personas}
    li = len(layers) // 2
    M = np.stack([behav[p][li] for p in personas])
    C = H.cosine_matrix(M)
    coarse = {p: H.coarse_category(ds.persona_category.get(p, "")) for p in personas}
    try:
        from scipy.cluster.hierarchy import linkage, fcluster
        from scipy.spatial.distance import squareform
        D = 1.0 - C; np.fill_diagonal(D, 0.0); D = (D + D.T) / 2
        Z = linkage(squareform(D, checks=False), method="average")
        labels = fcluster(Z, t=4, criterion="maxclust")
    except Exception as e:
        from sklearn.cluster import KMeans
        print("scipy unavailable, KMeans fallback:", e)
        labels = KMeans(n_clusters=min(4, len(personas)), n_init=10, random_state=cfg.seed).fit_predict(H.unit(M))
    cluster = dict(zip(personas, [int(x) for x in labels]))
    # do the confounds land together, separate from the historian control?
    conf = [p for p in personas if coarse[p] == "confound"]
    hist = [p for p in personas if "historian" in p]
    print("confound personas:", conf, "-> clusters", [cluster[p] for p in conf])
    print("historian control:", hist, "-> clusters", [cluster[p] for p in hist])
    H.save_npz(H.art_dir(cfg, "exp13") / "behavior_clustering.npz", personas=np.array(personas),
               cosine=C, labels=np.array(labels), rep_layer=np.array([layers[li]]))
    H.save_json({"personas": personas, "cluster": cluster, "coarse": coarse, "rep_layer": layers[li]},
                H.art_dir(cfg, "exp13") / "exp13.json")
    return {"personas": personas, "cosine": C, "cluster": cluster, "coarse": coarse, "rep_layer": layers[li]}


def plot_exp13(res):
    if "cosine" not in res:
        return
    personas = res["personas"]
    order = sorted(range(len(personas)), key=lambda i: (res["cluster"][personas[i]], res["coarse"][personas[i]]))
    labs = [f"{personas[i]} [{res['coarse'][personas[i]]}|c{res['cluster'][personas[i]]}]" for i in order]
    M = res["cosine"][np.ix_(order, order)]
    H.plot_heatmap(M, xticks=labs, yticks=labs, cmap="coolwarm", vmin=-1, vmax=1,
                   title=f"exp13: persona-behavior cosine, clustered (layer {res['rep_layer']})")


run_cell("exp13", plot_exp13)

## exp14 — Steer / color / project out (causal; runs last)

Re-runs forward passes with residual edits on the candidate self-rec direction
(exp02): **steer** (add it at the decision token, sweep layer × multiplier) and
**project out** (zero it across all positions). Measures the change in
authorship judgment (prob on the correct A/B) vs unsteered. Projecting out the
self-rec direction and watching recognition collapse is the primary causal
result.

**Requires** `cfg.enable_steering=True`, a GPU with the model, and the local eval
trial table (`results/text_evaluations/<task>/<run_name>/*.jsonl`) for the exact
prompts. Reuses the eval's model/hook infrastructure (`HFBackend`,
`core.activation_capture.decoder_layers`) — nothing is rebuilt.

In [ ]:
@H.register("exp14", "Steer / color / project out (causal)")
def exp14(ds, cfg, store):
    if not cfg.enable_steering:
        print("exp14 is causal: set cfg.enable_steering=True on a GPU box with the model.")
        return {"skipped": True, "reason": "enable_steering is False"}
    e2 = dep(store, "exp02"); pooled, layers, case = e2["pooled_read_dir"], e2["layers"], e2["case"]
    if pooled is None:
        return {"skipped": True, "reason": "no self-rec direction from exp02"}
    tt = H.load_trial_table(cfg)
    if tt.empty:
        print("no local trial table under results/text_evaluations/<task>/<run_name>/ — needed for prompts.")
        return {"skipped": True, "reason": "no trial table"}
    from core.backends.hf_backend import HFBackend
    from core.run_utils import resolve_dtype
    backend = HFBackend(cfg.model_name, torch_dtype=resolve_dtype(None))

    sub = tt[(tt["case_id"] == case)].dropna(subset=["prompt_text"])
    sub = sub.head(cfg.dry_run_max_per_cell if cfg.dry_run else 60)
    if sub.empty:
        return {"skipped": True, "reason": f"no {case} rows in trial table"}
    baseline = []
    for r in sub.itertuples(index=False):
        pr = H.steered_choice_probs(backend, _messages_from_row(r), ["A", "B"], [])
        baseline.append(pr.get(r.correct_answer, 0.0))
    base_mean = float(np.mean(baseline))

    alphas = [-8.0, -4.0, 4.0, 8.0]
    steer_rows, proj_rows = [], []
    for li, l in enumerate(layers):
        for alpha in alphas:
            ds_pc = []
            for r in sub.itertuples(index=False):
                edits = [{"layer": l, "vec": pooled[li], "alpha": alpha, "mode": "add",
                          "positions": (lambda seq: [seq - 1])}]  # steer the decision token
                pr = H.steered_choice_probs(backend, _messages_from_row(r), ["A", "B"], edits)
                ds_pc.append(pr.get(r.correct_answer, 0.0))
            steer_rows.append({"layer": l, "alpha": alpha, "mean_prob_correct": float(np.mean(ds_pc)),
                               "delta_vs_baseline": float(np.mean(ds_pc) - base_mean), "n": len(ds_pc)})
        ds_po = []
        for r in sub.itertuples(index=False):
            edits = [{"layer": l, "vec": pooled[li], "alpha": 0.0, "mode": "project_out", "positions": None}]
            pr = H.steered_choice_probs(backend, _messages_from_row(r), ["A", "B"], edits)
            ds_po.append(pr.get(r.correct_answer, 0.0))
        proj_rows.append({"layer": l, "mean_prob_correct": float(np.mean(ds_po)),
                          "delta_vs_baseline": float(np.mean(ds_po) - base_mean), "n": len(ds_po)})
    steer = pd.DataFrame(steer_rows); proj = pd.DataFrame(proj_rows)
    H.save_df(steer, H.art_dir(cfg, "exp14") / "steer_sweep.parquet")
    H.save_df(proj, H.art_dir(cfg, "exp14") / "project_out_sweep.parquet")
    peak = steer.iloc[steer["delta_vs_baseline"].abs().idxmax()] if len(steer) else None
    print(f"baseline mean prob(correct) = {base_mean:.3f}")
    if peak is not None:
        print(f"peak steer effect: layer {int(peak.layer)}, alpha {peak.alpha} -> "
              f"delta {peak.delta_vs_baseline:+.3f}")
    return {"case": case, "baseline": base_mean, "steer": steer, "project_out": proj, "layers": layers}


def plot_exp14(res):
    if res.get("skipped"):
        print("exp14 skipped:", res.get("reason")); return
    proj = res["project_out"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].axhline(res["baseline"], ls="--", c="grey", lw=1, label="baseline")
    for alpha, g in res["steer"].groupby("alpha"):
        ax[0].plot(g["layer"], g["mean_prob_correct"], marker="o", label=f"add a={alpha}")
    ax[0].set_title("exp14: steer at decision token"); ax[0].set_xlabel("layer")
    ax[0].set_ylabel("mean prob(correct)"); ax[0].legend(fontsize=8)
    ax[1].axhline(res["baseline"], ls="--", c="grey", lw=1, label="baseline")
    ax[1].plot(proj["layer"], proj["mean_prob_correct"], marker="o", c="crimson", label="project out")
    ax[1].set_title("exp14: project out self-rec direction"); ax[1].set_xlabel("layer")
    ax[1].set_ylabel("mean prob(correct)"); ax[1].legend(fontsize=8)
    fig.tight_layout()


run_cell("exp14", plot_exp14)

## Run the whole suite

Runs every registered experiment in canonical order (exp01 + exp04 first; exp14
last, skipped unless `cfg.enable_steering`). Each result is cached in `STORE` so
later experiments consume earlier ones. Flip `cfg.dry_run = False` first for a
full run.

In [ ]:
if "ds" in globals() and ds is not None:
    STORE = H.run_suite(ds, cfg, store=STORE)
    print("\nfinished:", [k for k in H.ORDER if k in STORE and not (isinstance(STORE.get(k), dict) and STORE[k].get('skipped'))])